In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# =============================
# 1) Load data
# =============================
train_df = pd.read_csv("train_carbreakdown.csv")
test_df  = pd.read_csv("test_carbreakdown.csv")
sample_sub = pd.read_csv("sample_submission.csv")

TARGET = "breakdown_next_30_days"

# Separate target and features
y = train_df[TARGET]
X = train_df.drop(columns=[TARGET])

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_sub.shape)

print("\nTarget distribution (counts):")
print(y.value_counts())
print("\nTarget distribution (percent):")
print(y.value_counts(normalize=True).round(4))

# =============================
# 2) Train/Validation split
# =============================
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nSplit shapes:")
print("X_train:", X_train.shape, "| X_val:", X_val.shape)

# =============================
# 3) Preprocessing (handle missing + encode categories)
# =============================
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# =============================
# 4) Random Forest benchmark model
# =============================
rf = RandomForestClassifier(
    n_estimators=800,
    random_state=42,
    class_weight="balanced",
    max_features="sqrt",
    min_samples_leaf=2,
    n_jobs=-1
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf)
])

# =============================
# 5) Fit on train split
# =============================
model.fit(X_train, y_train)

# =============================
# 6) Validate (required for your report)
# =============================
y_pred = model.predict(X_val)

acc = accuracy_score(y_val, y_pred)
cm = confusion_matrix(y_val, y_pred)

print("\nValidation Accuracy:", round(acc, 4))
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_val, y_pred, digits=4))

# =============================
# 7) Fit on FULL training data (best model for Kaggle)
# =============================
model.fit(X, y)

# =============================
# 8) Predict on test set as CLASS LABELS (0/1) for Kaggle
#    IMPORTANT: Your Kaggle metric expects binary labels, not probabilities.
# =============================
test_pred = model.predict(test_df)

# =============================
# 9) Create submission file
# =============================
submission = sample_sub.copy()
submission[TARGET] = test_pred
submission.to_csv("submission.csv", index=False)

print("\n✅ Saved submission.csv (0/1 labels). Preview:")
print(submission.head())
print("\nRows in submission:", submission.shape[0])
print("Columns in submission:", submission.columns.tolist())

Train shape: (1050, 14)
Test shape: (450, 13)
Sample submission shape: (450, 2)

Target distribution (counts):
breakdown_next_30_days
0    874
1    176
Name: count, dtype: int64

Target distribution (percent):
breakdown_next_30_days
0    0.8324
1    0.1676
Name: proportion, dtype: float64

Split shapes:
X_train: (840, 13) | X_val: (210, 13)

Validation Accuracy: 0.8333

Confusion Matrix:
 [[175   0]
 [ 35   0]]

Classification Report:
               precision    recall  f1-score   support

           0     0.8333    1.0000    0.9091       175
           1     0.0000    0.0000    0.0000        35

    accuracy                         0.8333       210
   macro avg     0.4167    0.5000    0.4545       210
weighted avg     0.6944    0.8333    0.7576       210



c:\Users\maria\Downloads\car-breakdown-prediction\AI_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\maria\Downloads\car-breakdown-prediction\AI_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\maria\Downloads\car-breakdown-prediction\AI_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p


✅ Saved submission.csv (0/1 labels). Preview:
     id  breakdown_next_30_days
0  1356                       0
1   889                       0
2  1190                       0
3   122                       0
4  1101                       0

Rows in submission: 450
Columns in submission: ['id', 'breakdown_next_30_days']
